# AGILAB worker class paths in Colab

This notebook separates worker-class resolution from the app launch flow for the source-checkout path.

- `execution_pandas_project` resolves to `ExecutionPandasWorker` and the `PandasWorker` family.
- `flight_project` resolves to `FlightWorker` and the `PolarsWorker` family.
- `uav_relay_queue_project` is shown through `AgiEnv` path resolution, while `DagWorker` is shown separately from the shared core.

It clones the AGILAB repository, installs it in editable mode, and inspects built-in app source paths directly.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_worker_paths.ipynb)


In [ ]:
# Source-checkout Colab path: clone AGILAB and install editable packages.
import shutil
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/content/agilab")
if (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_ROOT, check=True)
else:
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ThalesGroup/agilab.git", str(REPO_ROOT)], check=True)

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "agilab", "agi-core", "agi-cluster", "agi-node", "agi-env"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-cache-dir",
    "-e", "/content/agilab/src/agilab/core/agi-env",
    "-e", "/content/agilab/src/agilab/core/agi-node",
    "-e", "/content/agilab/src/agilab/core/agi-cluster",
    "-e", "/content/agilab/src/agilab/core/agi-core",
    "-e", "/content/agilab",
], check=True)


In [ ]:
import importlib
import os
import sys
from pathlib import Path

os.environ["IS_SOURCE_ENV"] = "1"
os.environ["AGI_CLUSTER_ENABLED"] = "0"
os.environ.pop("IS_WORKER_ENV", None)

for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.")):
        del sys.modules[name]
importlib.invalidate_caches()

REPO_ROOT = Path("/content/agilab")
for entry in reversed([
    REPO_ROOT / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-env" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-node" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-cluster" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-core" / "src",
]):
    entry_str = str(entry)
    if entry_str not in sys.path:
        sys.path.insert(0, entry_str)

from agi_env import AgiEnv
from agi_node.dag_worker import DagWorker
from agi_node.pandas_worker import PandasWorker
from agi_node.polars_worker import PolarsWorker

REPO_ROOT = Path("/content/agilab")
BUILTIN_ROOT = REPO_ROOT / "src" / "agilab" / "apps" / "builtin"
for entry in reversed([
    BUILTIN_ROOT / "execution_pandas_project" / "src",
    BUILTIN_ROOT / "flight_project" / "src",
]):
    entry_str = str(entry)
    if entry_str not in sys.path:
        sys.path.insert(0, entry_str)

print("Repo root:", REPO_ROOT)
print("Builtin root:", BUILTIN_ROOT)
print("execution_pandas_project source:", (BUILTIN_ROOT / "execution_pandas_project" / "src").is_dir())
print("flight_project source:", (BUILTIN_ROOT / "flight_project" / "src").is_dir())
print("uav_relay_queue_project source:", (BUILTIN_ROOT / "uav_relay_queue_project" / "src").is_dir())

from execution_pandas_worker.execution_pandas_worker import ExecutionPandasWorker
from flight_worker.flight_worker import FlightWorker

def describe_app(app_name: str, imported_cls=None):
    env = AgiEnv(apps_path=BUILTIN_ROOT, app=app_name, verbose=0)
    print(f"\n=== {app_name} ===")
    print("target worker class:", getattr(env, "target_worker_class", None))
    print("worker source:", getattr(env, "worker_path", None))
    print("resolved worker target:", getattr(env, "target_worker", None))
    if imported_cls is not None:
        print("imported class:", imported_cls.__module__ + "." + imported_cls.__name__)
        print("MRO:", " -> ".join(cls.__name__ for cls in imported_cls.mro()[:4]))
    return env

describe_app("execution_pandas_project", ExecutionPandasWorker)
describe_app("flight_project", FlightWorker)
describe_app("uav_relay_queue_project")

print("\n=== shared DagWorker ===")
print("imported class:", DagWorker.__module__ + "." + DagWorker.__name__)
print("MRO:", " -> ".join(cls.__name__ for cls in DagWorker.mro()[:4]))
print("base worker families:", PandasWorker.__name__, "/", PolarsWorker.__name__)


## What this proves

- The AGILAB core packages can be installed directly from the cloned repository in a notebook runtime.
- The built-in app sources can be fetched by cloning the repository inside Colab.
- `AgiEnv` resolves the worker source path for each built-in app.
- `execution_pandas_project` binds to `ExecutionPandasWorker`, whose base family is `PandasWorker`.
- `flight_project` binds to `FlightWorker`, whose base family is `PolarsWorker`.
- `DagWorker` is available as the shared DAG execution base class.
- `uav_relay_queue_project` stays visible as a resolved app worker path, without pretending it is the same thing as `DagWorker`.

## Next step

After this class-path check works, use the launch notebooks if you want to run the apps end to end.
